# 01 — EDA, Qualidade de Dados e Reestruturação PEDE 2025
Análise exploratória longitudinal da base Passos Mágicos (2022–2025), auditoria de valores ausentes, verificação de esquema e leitura analítica da reestruturação metodológica.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import settings
from src.pipeline.data_validation import load_and_validate_pede

df_raw, validation_report = load_and_validate_pede()
print(f"Registros totais: {len(df_raw)} | Colunas: {df_raw.shape[1]}")
print("Anos presentes:", sorted(df_raw["ano_referencia"].dropna().unique().tolist()))
print("Relatório de validação:", validation_report)

## 1. Cobertura Temporal e Volume por Ciclo
Verificação da distribuição de estudantes/registos por ano de referência para avaliar continuidade longitudinal e possíveis quebras de série.

In [ ]:
cobertura_ano = (
    df_raw.groupby("ano_referencia", dropna=False)
    .size()
    .reset_index(name="total_registros")
    .assign(percentual=lambda d: (d["total_registros"] / len(df_raw) * 100).round(2))
)
display(cobertura_ano)

## 2. Auditoria de Valores Ausentes (Top 15 Colunas)
Identificação de lacunas de preenchimento que exigem tratamento específico ou cautela na engenharia de features.

In [ ]:
nulos_df = (
    pd.DataFrame({
        "qtd_nulos": df_raw.isna().sum(),
        "pct_nulos": (df_raw.isna().mean() * 100).round(2),
        "dtype": df_raw.dtypes.astype(str),
    })
    .sort_values(["pct_nulos", "qtd_nulos"], ascending=False)
)
display(nulos_df.head(15))

## 3. Estatísticas Descritivas Numéricas e Contexto PEDE 2025
Resumo quantitativo das variáveis numéricas e registro das implicações da reestruturação metodológica observada em 2025 para o desenho anti-leakage ( 
ightarrow t+1$).

In [ ]:
num_cols = df_raw.select_dtypes(include="number").columns.tolist()
if num_cols:
    display(df_raw[num_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3))
else:
    print("Nenhuma coluna numérica detectada diretamente.")